# Module 3 · Modern Transformer Architecture — From Attention to Mini-GPT

**Phase:** Deep Learning Foundations  
**Prerequisites:** NB15_01 (Neural Network Foundations), NB15_04 (Sequence Models)  
**Toolchain:** PyTorch  
**Objective:** Build every component of a modern transformer from scratch — Multi-Head Attention, GQA, RoPE, SwiGLU FFN, KV-Cache — and train a complete mini-GPT character-level language model.

---

## Why This Is the Most Important Architecture Notebook

Every major AI system in 2026 — GPT-4o, Llama 3, Gemini, Claude — is a transformer. Understanding the architecture at the implementation level is non-negotiable for AI engineers who need to:
- Fine-tune models (NB16_07) and understand what layers they're modifying
- Debug attention patterns and generation quality
- Choose between MHA, GQA, and MQA for deployment constraints
- Optimize inference (NB30) with KV-Cache and Flash Attention

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** You don't just 'use a transformer'. You understand the trade-offs: **RoPE** for context length extrapolation, **GQA** for multi-user inference throughput, and **Pre-Norm** for training stability in 100+ layer architectures.

**Mental Model:** Imagine a library where every book (token) has a post-it note.
- **Query:** Your search term.
- **Key:** The index on the shelf.
- **Value:** The actual text in the book.
- **GQA:** Sharing the same index/text for multiple search terms to save bookshelf space.
- **RoPE:** Giving every book a 'clock' that rotates based on its position, so the model 'feels' the distance between them.

**Common Junior Pitfall:** Recomputing the entire sequence's attention for every single token generated. Seniors use a **KV-Cache** to store past results, making generation linear in time rather than quadratic.

---


## 📑 Table of Contents

* [Why This Is the Most Important Architecture Notebook](#why-this-is-the-most-important-architecture-notebook)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Scaled Dot-Product Attention](#1--scaled-dot-product-attention)
* [2 · Multi-Head Attention (MHA)](#2--multi-head-attention-mha)
* [3 · Grouped-Query Attention (GQA)](#3--grouped-query-attention-gqa)
* [4 · Rotary Position Embeddings (RoPE)](#4--rotary-position-embeddings-rope)
* [5 · Feed-Forward Networks: GELU vs SwiGLU](#5--feed-forward-networks-gelu-vs-swiglu)
* [6 · Pre-Norm vs Post-Norm](#6--pre-norm-vs-post-norm)
* [7 · The Full Transformer Stack](#7--the-full-transformer-stack)
* [8 · Flash Attention: Making O(N^2) Practical](#8--flash-attention-making-on2-practical)
* [9 · The KV-Cache](#9--the-kv-cache)
* [10 · Mini-GPT: Training a Transformer from Scratch](#10--mini-gpt-training-a-transformer-from-scratch)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Practice](#-practical-practice)
* [🎯 Summary & Key Takeaways](#-summary--key-takeaways)


In [ ]:
# Cell 0 — Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

print(f'PyTorch {torch.__version__}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')


---
## 1 · Scaled Dot-Product Attention

The core operation of every transformer. Given queries Q, keys K, and values V:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### 🎓 Socratic Deep Check
> *"Why do we divide by $\sqrt{d_k}$? If $d_k=1024$ and we didn't scale, the dot products would have variance ~1024. The softmax would become extremely 'sharp' (one-hot), crushing gradients to near-zero. Scaling keeps dot products in a range where softmax gradients flow."*

### The Causal Mask
For autoregressive (decoder) models, token $i$ must not attend to tokens $j > i$. We fill those positions with $-\infty$ before softmax, making those attention weights exactly 0.


In [ ]:
# Cell 1 — Scaled dot-product attention from scratch
def scaled_dot_product_attention(q, k, v, mask=None, dropout=None):
    """Core attention: softmax(QK^T / sqrt(d_k)) * V
    
    Args:
        q: (batch, heads, seq_len, head_dim)
        k: (batch, heads, seq_len, head_dim)
        v: (batch, heads, seq_len, head_dim)
        mask: causal mask (optional)
    Returns:
        output: (batch, heads, seq_len, head_dim)
        attention_weights: (batch, heads, seq_len, seq_len)
    """
    d_k = q.size(-1)
    # 1. Dot product: how relevant is each key to each query?
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    
    # 2. Causal mask: prevent attending to future tokens
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # 3. Softmax: convert scores to probabilities
    attn_weights = F.softmax(scores, dim=-1)
    
    if dropout is not None:
        attn_weights = dropout(attn_weights)
    
    # 4. Weighted sum of values
    return torch.matmul(attn_weights, v), attn_weights

# Demo: 4 tokens attending to each other
torch.manual_seed(42)
seq_len, d_k = 4, 8
q = torch.randn(1, 1, seq_len, d_k)
k = torch.randn(1, 1, seq_len, d_k)
v = torch.randn(1, 1, seq_len, d_k)

# Create causal mask
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)

out, weights = scaled_dot_product_attention(q, k, v, mask=causal_mask)
print('Causal attention weights (each row sums to 1):')
print(weights[0, 0].detach().numpy().round(3))
print()
print('Key insight: Row 0 can only attend to token 0.')
print('            Row 3 can attend to tokens 0,1,2,3.')
print('            This is how autoregressive generation works.')


---
## 2 · Multi-Head Attention (MHA)

A single attention head can only focus on one kind of relationship. **Multi-Head Attention** runs $H$ parallel attention operations, each learning different patterns:

- Head 1 might learn syntactic relationships (subject-verb)
- Head 2 might learn co-reference ("it" -> "the cat")
- Head 3 might learn positional patterns (nearby tokens)

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_H) W^O$$

where each $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$


In [ ]:
# Cell 2 — Standard Multi-Head Attention
class MultiHeadAttention(nn.Module):
    """Standard MHA: H query heads, H key heads, H value heads."""
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        
        # Projections: input -> Q, K, V
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x, mask=None):
        B, L, D = x.shape
        
        # Project and reshape: (B, L, D) -> (B, H, L, head_dim)
        q = self.q_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention for all heads in parallel
        out, _ = scaled_dot_product_attention(q, k, v, mask)
        
        # Concat heads and project: (B, H, L, head_dim) -> (B, L, D)
        out = out.transpose(1, 2).reshape(B, L, D)
        return self.o_proj(out)

# Test
mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)  # batch=2, seq=10, dim=64
out = mha(x)
print(f'MHA: input {x.shape} -> output {out.shape}')
print(f'  Heads: {mha.num_heads}, Head dim: {mha.head_dim}')
params = sum(p.numel() for p in mha.parameters())
print(f'  Parameters: {params:,}  (4 projection matrices)')


---
## 3 · Grouped-Query Attention (GQA)

MHA stores H key and value heads per token. For long sequences, this KV-cache dominates GPU memory. GQA shares KV heads across groups of query heads:

| Variant | Q heads | KV heads | KV memory | Quality |
|---------|:---:|:---:|:---:|:---:|
| **MHA** | H | H | Baseline | Best |
| **GQA** | H | H/G | 1/G of MHA | Near-MHA |
| **MQA** | H | 1 | 1/H of MHA | Lower |

**Production Signal:** Llama 3 8B uses H=32, G=8 (4 KV heads per 32 Q heads), reducing KV-cache by 4x.


In [ ]:
# Cell 3 — Grouped-Query Attention
class GroupedQueryAttention(nn.Module):
    """GQA: fewer KV heads shared across query head groups."""
    
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = d_model // num_heads
        self.kv_group_size = num_heads // num_kv_heads
        
        self.q_proj = nn.Linear(d_model, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, d_model, bias=False)
    
    def forward(self, x, mask=None):
        B, L, _ = x.shape
        q = self.q_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, L, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, L, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # GQA: expand KV heads to match Q heads via repeat
        if self.num_kv_heads != self.num_heads:
            k = torch.repeat_interleave(k, self.kv_group_size, dim=1)
            v = torch.repeat_interleave(v, self.kv_group_size, dim=1)
        
        out, _ = scaled_dot_product_attention(q, k, v, mask)
        out = out.transpose(1, 2).reshape(B, L, -1)
        return self.o_proj(out)

# Compare parameter counts
d = 256
mha = MultiHeadAttention(d, 8)
gqa = GroupedQueryAttention(d, num_heads=8, num_kv_heads=2)

mha_params = sum(p.numel() for p in mha.parameters())
gqa_params = sum(p.numel() for p in gqa.parameters())
print(f'MHA (8 Q, 8 KV heads): {mha_params:,} params')
print(f'GQA (8 Q, 2 KV heads): {gqa_params:,} params')
print(f'KV parameter reduction: {1 - gqa_params/mha_params:.0%}')
print(f'\nAt 32K context, GQA KV-cache is 4x smaller = 4x higher batch size')


---
## 4 · Rotary Position Embeddings (RoPE)

Original transformers used absolute sine waves added to embeddings. RoPE instead **rotates** vectors in complex space, encoding relative position directly into the dot product.

For dimension pair $(2i, 2i+1)$ at position $m$:
$$R_m = \\begin{pmatrix} \\cos(m\\theta_i) & -\\sin(m\\theta_i) \\\\ \\sin(m\\theta_i) & \\cos(m\\theta_i) \\end{pmatrix}$$

**The Critical Property:** $\\langle R_m q, R_n k \\rangle = \\langle q, R_{n-m} k \\rangle$ — the dot product depends only on the **distance** $(n-m)$, not absolute positions. This enables context length extrapolation.

**Production Signal:** RoPE is used by Llama, Mistral, and Qwen. NTK-aware scaling extends trained 8K context to 128K+ at inference.


In [ ]:
# Cell 4 — Rotary Position Embeddings
def apply_rotary_emb(x, seq_len, base=10000):
    """Apply RoPE to input tensor.
    Args: x shape (batch, heads, seq_len, head_dim)
    """
    d = x.shape[-1]
    # Compute rotation frequencies
    theta = 1.0 / (base ** (torch.arange(0, d, 2).float() / d))
    positions = torch.arange(seq_len).float()
    angles = positions.unsqueeze(-1) * theta.unsqueeze(0)  # (seq, d/2)
    
    cos = torch.cos(angles).unsqueeze(0).unsqueeze(0)  # (1, 1, seq, d/2)
    sin = torch.sin(angles).unsqueeze(0).unsqueeze(0)
    
    # Split into even/odd pairs and rotate
    x1, x2 = x[..., 0::2], x[..., 1::2]
    x_rotated = torch.stack([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)
    return x_rotated.flatten(-2)

# Verify the relative position property
torch.manual_seed(42)
q = torch.randn(1, 1, 5, 8)  # 5 positions
k = torch.randn(1, 1, 5, 8)

q_rot = apply_rotary_emb(q, 5)
k_rot = apply_rotary_emb(k, 5)

# Dot product between position 1 and 3 (distance = 2)
dot_1_3 = (q_rot[0,0,1] * k_rot[0,0,3]).sum().item()
# Dot product between position 2 and 4 (also distance = 2)
dot_2_4 = (q_rot[0,0,2] * k_rot[0,0,4]).sum().item()

print(f'Dot product (pos 1, pos 3) distance=2: {dot_1_3:.4f}')
print(f'Dot product (pos 2, pos 4) distance=2: {dot_2_4:.4f}')
print(f'Same distance = similar dot products (RoPE encodes RELATIVE position)')


---
## 5 · Feed-Forward Networks: GELU vs SwiGLU

Every transformer block has an attention layer AND a feed-forward network (FFN). The FFN processes each token independently:

| FFN Type | Formula | Used By |
|----------|---------|--------|
| **Original** | `Linear(d, 4d) -> ReLU -> Linear(4d, d)` | GPT-2 |
| **GELU** | `Linear(d, 4d) -> GELU -> Linear(4d, d)` | BERT, GPT-3 |
| **SwiGLU** | `Linear(d, 4d/3*2) -> SiLU * gate -> Linear(4d/3, d)` | Llama, Mistral, Gemma |

### Why SwiGLU?

SwiGLU uses a **gating mechanism**: one linear projection produces the activations, another produces a gate that controls information flow. This consistently improves training loss compared to ReLU/GELU at the same parameter count.

The hidden dimension is typically $\\frac{8}{3}d$ (not $4d$) to keep parameter count equal to standard FFN.


In [ ]:
# Cell 5 — Feed-Forward Networks: Standard vs SwiGLU

class StandardFFN(nn.Module):
    """GPT-3 style: Linear -> GELU -> Linear"""
    def __init__(self, d_model, expansion=4):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_model * expansion)
        self.fc2 = nn.Linear(d_model * expansion, d_model)
    
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class SwiGLUFFN(nn.Module):
    """Llama-style: Gated Linear Unit with SiLU activation.
    Uses 8/3 * d_model hidden dim to match parameter count."""
    def __init__(self, d_model, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or int(8 * d_model / 3)
        # Round to nearest multiple of 64 for GPU efficiency
        hidden_dim = 64 * ((hidden_dim + 63) // 64)
        
        self.gate_proj = nn.Linear(d_model, hidden_dim, bias=False)  # Gate
        self.up_proj = nn.Linear(d_model, hidden_dim, bias=False)    # Up
        self.down_proj = nn.Linear(hidden_dim, d_model, bias=False)  # Down
    
    def forward(self, x):
        # SwiGLU: down(SiLU(gate(x)) * up(x))
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

# Compare
d = 256
std = StandardFFN(d)
swi = SwiGLUFFN(d)
std_p = sum(p.numel() for p in std.parameters())
swi_p = sum(p.numel() for p in swi.parameters())
x = torch.randn(1, 10, d)
print(f'Standard FFN: {std_p:,} params, output {std(x).shape}')
print(f'SwiGLU FFN:   {swi_p:,} params, output {swi(x).shape}')
print(f'\nSwiGLU has 3 matrices instead of 2, but uses smaller hidden dim.')
print(f'Result: similar param count, but consistently lower training loss.')


---
## 6 · Pre-Norm vs Post-Norm

- **Post-Norm (original 2017):** $x = \\text{LN}(x + \\text{Sublayer}(x))$
- **Pre-Norm (modern):** $x = x + \\text{Sublayer}(\\text{LN}(x))$

Pre-Norm leaves the **residual highway** untouched. Gradients flow directly through the identity path, making 100+ layer training stable without warmup tricks.

**Production Signal:** GPT-3, Llama, Mistral, and Gemma all use Pre-Norm with RMSNorm (a faster variant of LayerNorm that skips mean-centering).


---
## 7 · The Full Transformer Stack

Now we assemble everything into a complete decoder-only transformer:

```
Input tokens
     |
[Token Embedding + Positional Encoding]
     |
[TransformerBlock x N]
  |-- RMSNorm -> GQA Attention -> + residual
  |-- RMSNorm -> SwiGLU FFN -> + residual
     |
[RMSNorm]
     |
[Language Model Head (Linear -> vocab_size)]
     |
Logits -> next token probabilities
```


In [ ]:
# Cell 6 — Complete Transformer: Embedding -> N blocks -> LM head

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (faster than LayerNorm)."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    
    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

class TransformerBlock(nn.Module):
    """Single Llama-style transformer block: Pre-Norm + GQA + SwiGLU."""
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, num_heads, num_kv_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLUFFN(d_model)
    
    def forward(self, x, mask=None):
        # Pre-Norm: normalize BEFORE sublayer, add residual
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

class MiniGPT(nn.Module):
    """Complete decoder-only transformer for character-level LM."""
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_kv_heads=2,
                 num_layers=4, max_seq_len=256):
        super().__init__()
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, num_kv_heads)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying: share embedding and output weights
        self.lm_head.weight = self.token_emb.weight
    
    def forward(self, x):
        B, L = x.shape
        # Causal mask
        mask = torch.tril(torch.ones(L, L, device=x.device)).unsqueeze(0).unsqueeze(0)
        
        h = self.token_emb(x) * math.sqrt(self.d_model)  # Scale embeddings
        h = apply_rotary_emb(h.unsqueeze(1), L).squeeze(1)  # Add RoPE  
        
        for block in self.blocks:
            h = block(h, mask)
        
        h = self.norm(h)
        logits = self.lm_head(h)  # (B, L, vocab_size)
        return logits
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]  # Truncate to context window
            logits = self(idx_cond)[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, 1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx

# Test the architecture
model = MiniGPT(vocab_size=128, d_model=128, num_heads=4, num_kv_heads=2, num_layers=4)
total_params = sum(p.numel() for p in model.parameters())
print(f'MiniGPT Architecture:')
print(f'  Layers: 4, Heads: 4 (2 KV), d_model: 128')
print(f'  Total parameters: {total_params:,}')
print(f'  Weight tying: embedding == lm_head (saves {128*128:,} params)')

# Forward pass test
x = torch.randint(0, 128, (2, 32))  # batch=2, seq=32
logits = model(x)
print(f'\n  Input:  {x.shape} (batch=2, seq=32)')
print(f'  Output: {logits.shape} (batch=2, seq=32, vocab=128)')


---
## 8 · Flash Attention: Making O(N^2) Practical

Standard attention computes the **full N x N attention matrix**, which for N=128K requires 128K x 128K x 2 bytes = **32GB** just for one layer's attention scores.

**Flash Attention** (Dao et al.) never materializes this matrix. Instead, it:

1. **Tiles** the Q, K, V matrices into small blocks that fit in GPU SRAM (fast on-chip memory)
2. Computes attention **block-by-block**, accumulating the softmax numerator and denominator incrementally
3. Never writes the N x N matrix to slow HBM (GPU main memory)

```
Standard Attention:                Flash Attention:
HBM (slow) <-> SRAM (fast)         HBM -> SRAM -> compute -> SRAM -> HBM
  Write full NxN matrix              Never write NxN matrix
  Memory: O(N^2)                     Memory: O(N)
  IO-bound                           Compute-bound (2-4x faster)
```

### Using Flash Attention in Practice

```python
# PyTorch 2.0+ has Flash Attention built in!
# It automatically activates when using:
F.scaled_dot_product_attention(q, k, v, is_causal=True)
# This is 2-4x faster and uses O(N) memory instead of O(N^2)
```

**Production Signal:** All major inference engines (vLLM, TensorRT-LLM, SGLang) use Flash Attention 2/3 by default.


---
## 9 · The KV-Cache

During autoregressive generation, each new token only needs to compute attention against ALL previous tokens. Without caching, this means re-processing the entire sequence for every new token.

**KV-Cache** stores computed K and V tensors from previous tokens:

```
Step 1: Compute Q1,K1,V1 -> Attend(Q1, [K1], [V1]) -> cache K1,V1
Step 2: Compute Q2,K2,V2 -> Attend(Q2, [K1,K2], [V1,V2]) -> cache K2,V2
Step 3: Compute Q3,K3,V3 -> Attend(Q3, [K1,K2,K3], [V1,V2,V3]) -> cache K3,V3
```

Each step: **O(1)** new computation instead of re-processing everything.


In [ ]:
# Cell 7 — KV-Cache memory and speedup analysis

def kv_cache_analysis(num_layers, num_kv_heads, head_dim, seq_len, dtype_bytes=2):
    """Calculate KV-cache memory for a given model config."""
    # Each layer stores K and V tensors
    # K/V shape per layer: (batch, num_kv_heads, seq_len, head_dim)
    per_token_per_layer = 2 * num_kv_heads * head_dim * dtype_bytes  # 2 for K and V
    total_bytes = per_token_per_layer * num_layers * seq_len
    return total_bytes

# Compare cache sizes for different architectures
configs = {
    'Llama 3 8B (GQA)':  (32, 8, 128, 8192),    # 32 layers, 8 KV heads
    'Llama 3 8B (MHA)':  (32, 32, 128, 8192),   # Hypothetical with full MHA
    'Llama 3 70B (GQA)': (80, 8, 128, 8192),    # 80 layers, 8 KV heads
}

print('KV-Cache Memory Analysis (fp16, single user):')
print('=' * 60)
for name, (layers, kv_heads, head_dim, seq) in configs.items():
    mem = kv_cache_analysis(layers, kv_heads, head_dim, seq)
    print(f'  {name:25s}: {mem/1024**2:7.1f} MB  ({mem/1024**3:.2f} GB)')

# Speedup analysis
print(f'\nGeneration Speedup (seq_len=200):')
seq = 200
naive = sum(range(seq))  # Recompute all previous at each step
cached = seq             # Only compute new token
print(f'  Naive (no cache): {naive:,} attention operations')
print(f'  KV-Cache:         {cached:,} attention operations')
print(f'  Speedup:          {naive/cached:.0f}x')


---
## 10 · Mini-GPT: Training a Transformer from Scratch

Now let's train our full transformer on a character-level language modeling task. The model will learn to predict the next character given a sequence of characters.

### Training Dynamics

| Phase | What Happens | When |
|-------|-------------|------|
| Warmup | Learning rate ramps from 0 to peak | First 5-10% of steps |
| Stable training | Loss decreases smoothly | 10-90% of steps |
| Loss spikes | Random sharp increases (recover quickly) | Occasionally |
| Convergence | Loss plateaus | Final 10% |


In [ ]:
# Cell 8 — Train MiniGPT on character-level text
import torch
import torch.nn.functional as F

# Generate training data: repeating patterns for fast learning
text = """The transformer architecture revolutionized AI. Attention is all you need. 
Multi-head attention lets the model focus on different parts of the input simultaneously.
Position embeddings encode where each token appears in the sequence.
The feed-forward network processes each token independently after attention.
Layer normalization stabilizes training of deep networks.
Residual connections allow gradients to flow through many layers.
""" * 50  # Repeat to get enough training data

# Character-level tokenizer
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
print(f'Training data: {len(data):,} characters, {vocab_size} unique tokens')

# Create model
model = MiniGPT(vocab_size=vocab_size, d_model=64, num_heads=4,
                num_kv_heads=2, num_layers=3, max_seq_len=64)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {total_params:,} parameters')

# Training loop
model.train()
batch_size, seq_len = 32, 64
losses = []

for step in range(300):
    # Random batch of sequences
    idx = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in idx])
    y = torch.stack([data[i+1:i+seq_len+1] for i in idx])
    
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
    optimizer.step()
    
    losses.append(loss.item())
    if step % 50 == 0:
        print(f'  Step {step:3d}: loss = {loss.item():.4f}')

print(f'\nFinal loss: {losses[-1]:.4f} (random baseline: {math.log(vocab_size):.4f})')

# Generate text
model.eval()
prompt = 'The transformer'
prompt_ids = torch.tensor([[char_to_idx[c] for c in prompt]])
generated = model.generate(prompt_ids, max_new_tokens=100, temperature=0.7)
output = ''.join([idx_to_char[i.item()] for i in generated[0]])
print(f'\nGenerated text:')
print(f'  "{output}"')


---
## ✅ Knowledge Check

### Q1: Why GQA over MHA?
<details><summary>Answer</summary>

GQA reduces KV-cache memory by the group factor G. Since inference is memory-bandwidth-bound (not compute-bound), smaller KV-cache = larger batch sizes = higher throughput for serving multiple users. Quality loss is minimal.
</details>

### Q2: Why SwiGLU over GELU?
<details><summary>Answer</summary>

SwiGLU's gating mechanism (SiLU(Wx) * Vx) allows the network to learn which information to pass through. Empirically, it achieves lower training loss at the same parameter count. The hidden dimension is adjusted (8d/3 vs 4d) to match total parameters.
</details>

### Q3: Flash Attention memory
<details><summary>Answer</summary>

Standard attention materializes the full NxN scores matrix (O(N^2) memory). Flash Attention tiles the computation, processing small blocks in fast SRAM and never writing the full matrix to HBM. Result: O(N) memory, 2-4x faster on modern GPUs.
</details>

### Q4: Weight tying
Why does MiniGPT share weights between the embedding and the LM head?
<details><summary>Answer</summary>

The embedding maps token IDs to vectors. The LM head maps vectors back to token IDs. These are inverse operations, so sharing weights (a) reduces parameters significantly and (b) improves training by coupling the input and output representations.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Modify `scaled_dot_product_attention` to print attention weights before and after masking. Observe how the causal mask zeroes out future positions.
2. Implement a 2D rotation matrix and verify it preserves vector norms.

### Tier 2: Intermediate
1. Calculate KV-cache memory for Llama 3 70B at 32K context, fp16. How much VRAM?
2. Replace SwiGLU with standard GELU FFN in MiniGPT. Train both and compare final loss.
3. Implement `F.scaled_dot_product_attention` (PyTorch 2.0+) as a drop-in replacement.

### Tier 3: Advanced
1. Implement the Flash Attention tiling algorithm conceptually: process QKV in blocks of 32 tokens, accumulating softmax online (log-sum-exp trick).
2. Add a proper KV-Cache to MiniGPT's generate function. Measure generation speedup for 200-token sequences.
3. Implement NTK-aware RoPE scaling: multiply the base frequency by a factor to extend context window beyond training length.


---
## 🎯 Summary & Key Takeaways

| Component | What | Used By |
|-----------|------|--------|
| Scaled dot-product attention | Core QKV mechanism | All transformers |
| Multi-Head Attention | Parallel attention heads | GPT-2, BERT |
| Grouped-Query Attention | Shared KV heads (4x less cache) | Llama 3, Mistral |
| RoPE | Relative position via rotation | Llama, Qwen, Mistral |
| SwiGLU FFN | Gated feed-forward (better loss) | Llama, Gemma |
| Pre-Norm + RMSNorm | Stable deep training | All modern LLMs |
| Flash Attention | O(N) memory attention | vLLM, TRT-LLM |
| KV-Cache | O(1) per-step generation | All inference engines |
| Weight tying | Shared embedding/LM head | Most LLMs |

**Connections:** Sequence modeling → NB15_04 (RNNs/LSTMs) | NLP applications → NB16_01+ | Fine-tuning → NB16_07 (LoRA/QLoRA) | Inference optimization → NB30 | MoE variant → NB15_08

**Next →** `15_07_graph_neural_networks.ipynb` — Graph Neural Networks
